# Recover Corpus Metadata

Decodes the pre-tokenized corpus (huikang's `04-08-16-14` snapshot) to recover, per item:
- `problem_category`, `problem_id`, `answer` — by joining on `train.parquet`
- `recovered_cot` — the `<think>...</think>` content decoded from token ids
- `mask_segments` — each masked/trainable span with label, char offsets, and textual clues
- Flat boundary columns: `mask_begin_clue`, `mask_end_clue`, `trainable_begin_clue`

Also augments `train.parquet` with `used_in_training` and `n_corpus_variants` to flag the
~1670 samples that were NOT used in training.

**Corpus layout expected:**
```
<CORPUS_ROOT>/
  logprobs/index.jsonl          # {problem_id, category, num_loss_tokens, ...} per item
  tokens/{folder}/synthetic.json  # {tokens: [...], mask: [...]}  (folder name = problem_id)
```

**Outputs:**
- `recovered_corpus_metadata.parquet` — one row per corpus folder
- `train_with_coverage.parquet` — train.parquet + coverage columns

## Cell 1 — Configuration

In [ ]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# ── Corpus root (the sft/04-08-16-14 directory) ───────────────────────────────
# Update the Kaggle path after uploading the dataset snapshot.
CORPUS_ROOT_KAGGLE = "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/nemotron-master/training/sft/04-08-16-14"
CORPUS_ROOT_LOCAL  = "FILL_IN_LOCAL_PATH/04-08-16-14"  # ← set this for local runs
CORPUS_ROOT = CORPUS_ROOT_KAGGLE if IS_KAGGLE else CORPUS_ROOT_LOCAL

TOKENS_DIR  = os.path.join(CORPUS_ROOT, "tokens")
INDEX_PATH  = os.path.join(CORPUS_ROOT, "logprobs", "index.jsonl")

# ── Ground-truth train table ──────────────────────────────────────────────────
TRAIN_PARQUET_KAGGLE = "/kaggle/input/datasets/zosov07/oscar-analysis-inputs/train.parquet"
TRAIN_PARQUET_LOCAL  = "inputs/data/train.parquet"
TRAIN_PARQUET = TRAIN_PARQUET_KAGGLE if IS_KAGGLE else TRAIN_PARQUET_LOCAL

# ── Tokenizer ─────────────────────────────────────────────────────────────────
TOKENIZER_KAGGLE_MODEL = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
TOKENIZER_HF_FALLBACK  = "unsloth/Nemotron-3-Nano-30B-A3B"

# ── Outputs ───────────────────────────────────────────────────────────────────
OUT_DIR = "/kaggle/working" if IS_KAGGLE else "/tmp/corpus_recovery"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CORPUS_PARQUET   = os.path.join(OUT_DIR, "recovered_corpus_metadata.parquet")
OUT_COVERAGE_PARQUET = os.path.join(OUT_DIR, "train_with_coverage.parquet")

# ── Category normalisation map (corpus label → train.parquet label) ───────────
# Huikang's corpus uses finer-grained subtask labels (_deduce/_guess) and a
# separate -dN variant type — all map to the same coarse train.parquet category.
CATEGORY_MAP = {
    "bit_manipulation":        "Bit Manipulation",
    "cipher":                  "Cipher",
    "unit_conversion":         "Unit conversion",
    "numeral":                 "Numeral",
    "gravity":                 "Gravity",
    "cryptarithm_deduce":      "Cryptarithm (Symbolic Equations)",
    "cryptarithm_guess":       "Cryptarithm (Symbolic Equations)",
    "equation_numeric":        "Equation Numeric",
    "equation_numeric_deduce": "Equation Numeric",
    "equation_numeric_guess":  "Equation Numeric",
}

# ── Ops boilerplate opening marker (mask boundary clue) ───────────────────────
OPS_BOILERPLATE_MARKER = "Let's assume the following set of available operations"

# ── Optional row limit for quick dry-runs (set None to process all) ──────────
LIMIT = None  # e.g. 50 for smoke-test

# ── Clue window: number of chars to preview at span boundaries ────────────────
CLUE_CHARS = 80

print(f"IS_KAGGLE    : {IS_KAGGLE}")
print(f"CORPUS_ROOT  : {CORPUS_ROOT}")
print(f"TOKENS_DIR   : {TOKENS_DIR}")
print(f"INDEX_PATH   : {INDEX_PATH}")
print(f"TRAIN_PARQUET: {TRAIN_PARQUET}")
print(f"OUT_DIR      : {OUT_DIR}")
print(f"LIMIT        : {LIMIT}")

## Cell 2 — Install missing packages

In [ ]:
import importlib, subprocess, sys

def _ensure(package, import_name=None):
    import_name = import_name or package
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f"Installing {package} ...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

_ensure("transformers")
_ensure("pandas")
_ensure("tqdm")
_ensure("pyarrow")
if IS_KAGGLE:
    _ensure("kagglehub")

print("All packages available.")

## Cell 3 — Load tokenizer

In [ ]:
from transformers import AutoTokenizer

if IS_KAGGLE:
    import kagglehub
    tokenizer_path = kagglehub.model_download(TOKENIZER_KAGGLE_MODEL)
    print(f"Tokenizer downloaded to: {tokenizer_path}")
else:
    tokenizer_path = TOKENIZER_HF_FALLBACK
    print(f"Using tokenizer: {tokenizer_path}")

tok = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)
print(f"Vocab size: {tok.vocab_size:,}")
print(f"Chat template present: {'yes' if tok.chat_template else 'NO — will fail'}")

# ── Round-trip sanity check ───────────────────────────────────────────────────
# Encodes a known string and decodes it back; if the result doesn't match, the
# tokenizer is wrong and all subsequent decodes will be garbage.
_probe = "Hello, world! 42"
_probe_ids = tok.encode(_probe)
_probe_back = tok.decode(_probe_ids, skip_special_tokens=True).strip()
assert _probe_back == _probe, (
    f"Tokenizer round-trip FAILED: encoded then decoded '{_probe}' → '{_probe_back}'\n"
    "Wrong tokenizer loaded — all token→text decodes will be wrong."
)
print("Tokenizer round-trip sanity check: OK")

## Cell 4 — Load train.parquet and index.jsonl

In [ ]:
import json
import pandas as pd

# ── Ground-truth train table ──────────────────────────────────────────────────
train_df = pd.read_parquet(TRAIN_PARQUET)
print(f"train.parquet: {len(train_df):,} rows | cols: {list(train_df.columns)}")
assert train_df["problem_id"].nunique() == len(train_df), \
    "Duplicate problem_ids in train.parquet — join key is not unique!"
train_lookup = train_df.set_index("problem_id").to_dict(orient="index")
# {problem_id: {prompt, answer, problem_category}}

print(f"\nCategory distribution in train.parquet:")
print(train_df["problem_category"].value_counts().to_string())

# ── Corpus index.jsonl ────────────────────────────────────────────────────────
index_records: dict[str, dict] = {}  # folder_name → {category, num_loss_tokens, step, ...}
with open(INDEX_PATH) as f:
    for line in f:
        rec = json.loads(line)
        pid = rec["problem_id"]
        if pid in index_records:
            # Multiple entries per problem (e.g. multi-epoch) — keep the last.
            pass
        index_records[pid] = rec

print(f"\nindex.jsonl: {len(index_records):,} unique problem_ids")

# Discover category labels actually present in the corpus
corpus_categories = set(r["category"] for r in index_records.values())
print(f"Corpus category labels: {sorted(corpus_categories)}")
unknown_cats = corpus_categories - set(CATEGORY_MAP)
if unknown_cats:
    print(f"⚠️  UNMAPPED categories (add to CATEGORY_MAP): {unknown_cats}")
else:
    print("All corpus categories have a mapping. ✓")

# ── Glob tokens/ folders ──────────────────────────────────────────────────────
corpus_folders = sorted(
    d for d in os.listdir(TOKENS_DIR)
    if os.path.isdir(os.path.join(TOKENS_DIR, d))
       and os.path.isfile(os.path.join(TOKENS_DIR, d, "synthetic.json"))
)
print(f"\ntokens/ folders with synthetic.json: {len(corpus_folders):,}")

# Reconcile glob vs index.jsonl
_in_index_not_glob = set(index_records) - set(corpus_folders)
_in_glob_not_index = set(corpus_folders) - set(index_records)
if _in_index_not_glob:
    print(f"⚠️  In index.jsonl but no tokens/ folder ({len(_in_index_not_glob)}): "
          f"{list(_in_index_not_glob)[:5]}")
if _in_glob_not_index:
    print(f"⚠️  In tokens/ but missing from index.jsonl ({len(_in_glob_not_index)}): "
          f"{list(_in_glob_not_index)[:5]}")
if not _in_index_not_glob and not _in_glob_not_index:
    print("Glob and index.jsonl are in perfect agreement. ✓")

if LIMIT:
    corpus_folders = corpus_folders[:LIMIT]
    print(f"LIMIT active — processing {LIMIT} folders")

## Cell 5 — Decode + label helper

In [ ]:
import re
from typing import Optional


# ── Paraphrase strip ──────────────────────────────────────────────────────────
# Huikang's corpus has two variant suffix types:
#   -pN  (e.g. 00066667-p0) : duplicate of same prompt+cot, trained twice
#   -dN  (e.g. 0133bcec-d0) : different digit/symbol mapping for same base puzzle
# Both strip to the base train.parquet problem_id.
def strip_paraphrase(folder: str) -> str:
    """'00066667-p0' → '00066667',  '0133bcec-d3' → '0133bcec',  '00066667' → '00066667'."""
    return re.sub(r"-(p|d)\d+$", "", folder)


# ── Char-offset helper (avoid re-encoding the whole sequence repeatedly) ──────
def _tok_idx_to_char(full_text: str, tokens: list[int], tok_idx: int) -> int:
    """Return the char offset in full_text where token number tok_idx starts.

    Strategy: decode the prefix [0..tok_idx) and measure its length.  This is
    O(n) per call; for performance we cache a prefix-char-offset array once per item.
    """
    if tok_idx == 0:
        return 0
    if tok_idx >= len(tokens):
        return len(full_text)
    return len(tok.decode(tokens[:tok_idx], skip_special_tokens=False))


def _classify_run(decoded: str, mask_value: int) -> str:
    """Assign a semantic label to a run of equal mask values."""
    if mask_value == 1:
        return "trainable"
    # mask_value == 0
    if "<|im_start|>user" in decoded or "<|im_start|>system" in decoded:
        return "user_turn"
    if OPS_BOILERPLATE_MARKER in decoded:
        return "ops_boilerplate"
    if "<|im_start|>assistant" in decoded and "<think>" in decoded:
        return "assistant_header"
    # A masked run that starts/ends inside the assistant prefix but doesn't
    # contain the full ops list (e.g. the portion between the user turn and
    # the boilerplate in a split-mask scheme).
    return "other_masked"


def recover_item(folder: str) -> dict:
    """Decode one corpus item and return a metadata dict.

    Parameters
    ----------
    folder : str
        Subfolder name inside TOKENS_DIR (= the corpus problem_id).

    Returns
    -------
    dict with keys:
        corpus_folder, base_problem_id, is_paraphrase, variant_type,
        n_tokens, n_masked, n_trainable, n_mask_transitions,
        has_mask,
        recovered_prompt, recovered_cot, recovered_answer,
        mask_segments,          # list of span dicts
        mask_token_start,       # flat primary boundary columns
        mask_token_end,
        mask_char_start,
        mask_char_end,
        mask_begin_clue,
        mask_end_clue,
        trainable_begin_clue,
        loss_tokens_match,
        error,                  # None on success, error message string on failure
    """
    # Detect variant type from suffix
    _p = re.search(r"-(p|d)(\d+)$", folder)
    variant_type = _p.group(1) if _p else None  # 'p', 'd', or None

    result: dict = {
        "corpus_folder":    folder,
        "base_problem_id":  strip_paraphrase(folder),
        "is_paraphrase":    _p is not None,
        "variant_type":     variant_type,  # None=base, 'p'=duplicate, 'd'=digit-variant
        "error":            None,
    }

    try:
        # ── Load tokens (and mask if present) ────────────────────────────────
        syn_path = os.path.join(TOKENS_DIR, folder, "synthetic.json")
        with open(syn_path) as f:
            seg = json.load(f)

        tokens: list[int] = seg["tokens"]
        mask: Optional[list[int]] = seg.get("mask")  # None if absent
        has_mask = mask is not None

        result["n_tokens"]  = len(tokens)
        result["has_mask"]  = has_mask

        # ── Full-text decode ──────────────────────────────────────────────────
        full_text: str = tok.decode(tokens, skip_special_tokens=False)

        # ── Structural split (Nemotron chat markers) ─────────────────────────
        recovered_prompt  = ""
        recovered_cot     = ""
        recovered_answer  = ""

        # User turn: between <|im_start|>user\n and first <|im_end|>
        m_user = re.search(r"<\|im_start\|>user\n(.*?)<\|im_end\|>", full_text, re.DOTALL)
        if m_user:
            user_content = m_user.group(1)
            # Strip the boxed instruction if present (it's boilerplate appended in prediction mode)
            _boxed_instr = "Please put your final answer inside `\\boxed{}`."
            if _boxed_instr in user_content:
                recovered_prompt = user_content[:user_content.index(_boxed_instr)].rstrip("\n")
            else:
                recovered_prompt = user_content

        # CoT: inside <think>...</think>
        m_think = re.search(r"<think>(.*?)</think>", full_text, re.DOTALL)
        if m_think:
            recovered_cot = m_think.group(1).strip()

        # Answer: \boxed{...} after </think> (prediction mode)
        m_box = re.search(r"</think>.*?\\boxed\{(.*?)\}", full_text, re.DOTALL)
        if m_box:
            recovered_answer = m_box.group(1)

        result["recovered_prompt"] = recovered_prompt
        result["recovered_cot"]    = recovered_cot
        result["recovered_answer"] = recovered_answer

        # ── Mask-span analysis ────────────────────────────────────────────────
        if not has_mask:
            result.update({
                "n_masked": None, "n_trainable": None,
                "n_mask_transitions": None,
                "mask_segments": [],
                "mask_token_start": None, "mask_token_end": None,
                "mask_char_start": None,  "mask_char_end": None,
                "mask_begin_clue": None,  "mask_end_clue": None,
                "trainable_begin_clue": None,
                "loss_tokens_match": None,
            })
            return result

        assert len(mask) == len(tokens), \
            f"mask length {len(mask)} != tokens length {len(tokens)}"

        n_masked     = sum(1 for m in mask if m == 0)
        n_trainable  = sum(1 for m in mask if m == 1)
        transitions  = sum(1 for i in range(1, len(mask)) if mask[i] != mask[i - 1])

        result["n_masked"]           = n_masked
        result["n_trainable"]        = n_trainable
        result["n_mask_transitions"] = transitions

        # Walk mask runs
        mask_segments = []
        run_start = 0
        while run_start < len(mask):
            val = mask[run_start]
            run_end = run_start + 1
            while run_end < len(mask) and mask[run_end] == val:
                run_end += 1
            # [run_start, run_end)
            decoded_span = tok.decode(tokens[run_start:run_end], skip_special_tokens=False)
            label = _classify_run(decoded_span, val)

            # Char offsets in full_text for this span
            char_start = _tok_idx_to_char(full_text, tokens, run_start)
            char_end   = _tok_idx_to_char(full_text, tokens, run_end)

            # Textual clues: short previews at the boundary edges
            begin_clue = full_text[char_start : char_start + CLUE_CHARS]
            end_clue   = full_text[max(char_start, char_end - CLUE_CHARS) : char_end]

            mask_segments.append({
                "label":       label,
                "mask_value":  val,
                "tok_start":   run_start,
                "tok_end":     run_end,
                "char_start":  char_start,
                "char_end":    char_end,
                "begin_clue":  begin_clue,
                "end_clue":    end_clue,
            })
            run_start = run_end

        result["mask_segments"] = mask_segments

        # ── Flat primary-boundary columns ─────────────────────────────────────
        masked_segs    = [s for s in mask_segments if s["mask_value"] == 0]
        trainable_segs = [s for s in mask_segments if s["mask_value"] == 1]

        if masked_segs:
            first_m = masked_segs[0]
            last_m  = masked_segs[-1]
            result["mask_token_start"] = first_m["tok_start"]
            result["mask_token_end"]   = last_m["tok_end"]
            result["mask_char_start"]  = first_m["char_start"]
            result["mask_char_end"]    = last_m["char_end"]
            result["mask_begin_clue"]  = first_m["begin_clue"]
            result["mask_end_clue"]    = last_m["end_clue"]
        else:
            result.update({
                "mask_token_start": None, "mask_token_end": None,
                "mask_char_start":  None, "mask_char_end":  None,
                "mask_begin_clue":  None, "mask_end_clue":  None,
            })

        result["trainable_begin_clue"] = trainable_segs[0]["begin_clue"] if trainable_segs else None

        # ── Cross-check n_trainable vs index.jsonl ────────────────────────────
        idx_rec = index_records.get(folder, {})
        idx_loss_tokens = idx_rec.get("num_loss_tokens")
        if idx_loss_tokens is not None:
            result["loss_tokens_match"] = (n_trainable == idx_loss_tokens)
            if not result["loss_tokens_match"]:
                result["error"] = (
                    f"loss_tokens mismatch: decoded mask gives {n_trainable}, "
                    f"index.jsonl says {idx_loss_tokens}"
                )
        else:
            result["loss_tokens_match"] = None

    except Exception as exc:
        result["error"] = repr(exc)
        for k in [
            "n_tokens", "has_mask", "n_masked", "n_trainable", "n_mask_transitions",
            "recovered_prompt", "recovered_cot", "recovered_answer",
            "mask_segments", "mask_token_start", "mask_token_end",
            "mask_char_start", "mask_char_end",
            "mask_begin_clue", "mask_end_clue", "trainable_begin_clue",
            "loss_tokens_match",
        ]:
            result.setdefault(k, None)

    return result


# Quick smoke-test on the first folder
_sample = recover_item(corpus_folders[0])
print(f"Smoke-test folder     : {_sample['corpus_folder']}")
print(f"base_problem_id       : {_sample['base_problem_id']}")
print(f"is_paraphrase         : {_sample['is_paraphrase']}  variant_type={_sample['variant_type']}")
print(f"n_tokens              : {_sample['n_tokens']}")
print(f"n_masked / trainable  : {_sample['n_masked']} / {_sample['n_trainable']}")
print(f"n_mask_transitions    : {_sample['n_mask_transitions']}")
print(f"loss_tokens_match     : {_sample['loss_tokens_match']}")
print(f"error                 : {_sample['error']}")
print(f"recovered_prompt[:80] : {_sample['recovered_prompt'][:80]!r}")
print(f"recovered_cot[:80]    : {_sample['recovered_cot'][:80]!r}")
print(f"recovered_answer      : {_sample['recovered_answer']!r}")
print(f"mask_begin_clue       : {_sample['mask_begin_clue']!r}")
print(f"mask_end_clue         : {_sample['mask_end_clue']!r}")
print(f"trainable_begin_clue  : {_sample['trainable_begin_clue']!r}")
print(f"mask_segments count   : {len(_sample['mask_segments'] or [])}")
if _sample['mask_segments']:
    for s in _sample['mask_segments']:
        print(f"  [{s['mask_value']}] {s['label']:20s}  tok[{s['tok_start']}:{s['tok_end']}]  "
              f"char[{s['char_start']}:{s['char_end']}]  "
              f"begin={s['begin_clue'][:30]!r}  end={s['end_clue'][-30:]!r}")

## Cell 6 — Build per-item corpus table

In [ ]:
from tqdm.auto import tqdm

rows = []
for folder in tqdm(corpus_folders, desc="Decoding corpus"):
    row = recover_item(folder)

    # ── Attach index.jsonl fields ─────────────────────────────────────────────
    idx_rec = index_records.get(folder, {})
    row["category_corpus"]    = idx_rec.get("category", None)
    row["category_norm"]      = CATEGORY_MAP.get(row["category_corpus"], row["category_corpus"])
    row["num_loss_tokens_idx"] = idx_rec.get("num_loss_tokens", None)
    row["total_loss_idx"]      = idx_rec.get("total_loss", None)
    row["min_logprob_idx"]     = idx_rec.get("min_logprob", None)
    row["step_idx"]            = idx_rec.get("step", None)

    # ── Join train.parquet ────────────────────────────────────────────────────
    base_id = row["base_problem_id"]
    train_rec = train_lookup.get(base_id, {})
    row["problem_id"]         = base_id if base_id in train_lookup else None
    row["problem_category"]   = train_rec.get("problem_category", None)
    row["answer"]             = train_rec.get("answer", None)
    row["in_train_parquet"]   = base_id in train_lookup

    # ── Category cross-check ──────────────────────────────────────────────────
    if row["problem_category"] and row["category_norm"]:
        row["category_mismatch"] = row["problem_category"] != row["category_norm"]
    else:
        row["category_mismatch"] = None  # can't compare

    rows.append(row)

corpus_df = pd.DataFrame(rows)
print(f"\nCorpus table: {len(corpus_df):,} rows × {len(corpus_df.columns)} cols")

# Summary
n_errors = corpus_df["error"].notna().sum()
print(f"Errors          : {n_errors}")
print(f"Paraphrases     : {corpus_df['is_paraphrase'].sum()}")
print(f"Not in train    : {(~corpus_df['in_train_parquet']).sum()}")
print(f"Category mismatch: {corpus_df['category_mismatch'].sum(skipna=True)}")
print(f"loss_tokens_match=False: {(corpus_df['loss_tokens_match'] == False).sum()}")

if n_errors > 0:
    print("\n--- First few errors ---")
    print(corpus_df[corpus_df["error"].notna()][["corpus_folder", "error"]].head(5).to_string())

## Cell 7 — Coverage: flag unused train.parquet samples

In [ ]:
# ── Build used-base-id set from the FULL corpus (bypass LIMIT) ────────────────
# If LIMIT is set we still want the full used set for coverage, so we read
# all folders (fast — no decoding).
all_corpus_folders = sorted(
    d for d in os.listdir(TOKENS_DIR)
    if os.path.isdir(os.path.join(TOKENS_DIR, d))
       and os.path.isfile(os.path.join(TOKENS_DIR, d, "synthetic.json"))
)
used_base_ids = {strip_paraphrase(f) for f in all_corpus_folders}

# n_corpus_variants: how many corpus folders (incl. paraphrases) map to each train id
from collections import Counter
variant_counts = Counter(strip_paraphrase(f) for f in all_corpus_folders)

# Augment train_df
train_df["used_in_training"]  = train_df["problem_id"].isin(used_base_ids)
train_df["n_corpus_variants"] = train_df["problem_id"].map(variant_counts).fillna(0).astype(int)

n_used   = train_df["used_in_training"].sum()
n_unused = (~train_df["used_in_training"]).sum()
print(f"train.parquet total    : {len(train_df):,}")
print(f"used in training       : {n_used:,}")
print(f"NOT used in training   : {n_unused:,}")
print(f"\nUnused breakdown by category:")
print(
    train_df[~train_df["used_in_training"]]
    .groupby("problem_category").size()
    .rename("unused_count")
    .sort_values(ascending=False)
    .to_string()
)
print(f"\nPer-category usage rate:")
_cov = train_df.groupby("problem_category")["used_in_training"].agg(["sum", "count"])
_cov["pct_used"] = (_cov["sum"] / _cov["count"] * 100).round(1)
print(_cov.rename(columns={"sum": "used", "count": "total"}).to_string())

paraphrase_used = sum(1 for f in all_corpus_folders if re.search(r"-p\d+$", f))
print(f"\nCorpus folders total   : {len(all_corpus_folders):,}")
print(f"  paraphrase variants  : {paraphrase_used:,}")
print(f"  unique base puzzles  : {len(used_base_ids):,}")

## Cell 8 — Save outputs

In [ ]:
# Serialize mask_segments (list of dicts) as JSON string for parquet storage
import json as _json
corpus_df["mask_segments_json"] = corpus_df["mask_segments"].apply(
    lambda v: _json.dumps(v) if isinstance(v, list) else None
)
corpus_save = corpus_df.drop(columns=["mask_segments"])

corpus_save.to_parquet(OUT_CORPUS_PARQUET, index=False)
train_df.to_parquet(OUT_COVERAGE_PARQUET, index=False)

print(f"Saved: {OUT_CORPUS_PARQUET}")
print(f"       {OUT_COVERAGE_PARQUET}")
print(f"\nCorpus parquet shape  : {corpus_save.shape}")
print(f"Coverage parquet shape: {train_df.shape}")
print(f"\nCorpus columns: {list(corpus_save.columns)}")
print(f"Coverage columns: {list(train_df.columns)}")

## Cell 9 — Validation & spot-check

In [ ]:
import random

# ── Structural assertions ─────────────────────────────────────────────────────
# 1. Row count
if LIMIT is None:
    assert len(corpus_df) == len(all_corpus_folders), (
        f"Row count mismatch: {len(corpus_df)} vs {len(all_corpus_folders)} folders"
    )
    print(f"[OK] corpus_df has {len(corpus_df):,} rows (matches folder count)")

# 2. All corpus base_problem_ids are in train.parquet (orphan check)
orphans = set(corpus_df["base_problem_id"]) - set(train_df["problem_id"])
if orphans:
    print(f"⚠️  {len(orphans)} corpus base_problem_ids NOT found in train.parquet: "
          f"{list(orphans)[:10]}")
else:
    print(f"[OK] All corpus base_problem_ids exist in train.parquet")

# 3. Coverage arithmetic
if LIMIT is None:
    assert n_used + n_unused == len(train_df), \
        f"used ({n_used}) + unused ({n_unused}) != total ({len(train_df)})"
    print(f"[OK] used + unused = {n_used} + {n_unused} = {len(train_df)}")

# 4. loss_tokens_match rate
ltm = corpus_df["loss_tokens_match"]
n_checked = ltm.notna().sum()
n_ok      = (ltm == True).sum()
n_fail    = (ltm == False).sum()
print(f"[loss_tokens_match] checked={n_checked}, OK={n_ok}, FAIL={n_fail}")
if n_fail > 0:
    print("Mismatched items (first 5):")
    bad = corpus_df[ltm == False][["corpus_folder","n_trainable","num_loss_tokens_idx","error"]]
    print(bad.head(5).to_string())

# 5. category_mismatch
n_cat_mismatch = int(corpus_df["category_mismatch"].sum(skipna=True))
if n_cat_mismatch:
    print(f"⚠️  {n_cat_mismatch} category mismatches (corpus vs train.parquet):")
    bad_cat = corpus_df[corpus_df["category_mismatch"] == True]
    print(bad_cat[["corpus_folder","category_norm","problem_category"]].head(10).to_string())
else:
    print(f"[OK] category_mismatch = 0")

print()
# ── Human spot-check on 5 random items ───────────────────────────────────────
rng = random.Random(42)
_ok_rows = corpus_df[corpus_df["error"].isna()]
spot_idxs = rng.sample(list(_ok_rows.index), k=min(5, len(_ok_rows)))

for i, idx in enumerate(spot_idxs, 1):
    r = corpus_df.loc[idx]
    print(f"=== Spot-check {i}/{len(spot_idxs)} : {r['corpus_folder']} ===")
    print(f"  category (corpus/norm/train) : {r['category_corpus']} / {r['category_norm']} / {r['problem_category']}")
    print(f"  recovered_prompt[:100]       : {r['recovered_prompt'][:100]!r}")
    print(f"  recovered_cot[:120]          : {r['recovered_cot'][:120]!r}")
    print(f"  recovered_answer             : {r['recovered_answer']!r}")
    print(f"  train answer                 : {r['answer']!r}")
    print(f"  answers match                : {r['recovered_answer'] == r['answer']}")
    print(f"  n_tokens/masked/trainable    : {r['n_tokens']} / {r['n_masked']} / {r['n_trainable']}")
    print(f"  n_mask_transitions           : {r['n_mask_transitions']}")
    print(f"  mask_begin_clue              : {r['mask_begin_clue']!r}")
    print(f"  mask_end_clue                : {r['mask_end_clue']!r}")
    print(f"  trainable_begin_clue         : {r['trainable_begin_clue']!r}")
    # Print segment table
    segs = r["mask_segments"]
    if isinstance(segs, list):
        for s in segs:
            print(f"  [{s['mask_value']}] {s['label']:20s}  "
                  f"tok[{s['tok_start']}:{s['tok_end']}]  "
                  f"char[{s['char_start']}:{s['char_end']}]  "
                  f"begin={s['begin_clue'][:35]!r}")
    print()